# C3 model comparison on Google Colab (phase 1: validation only)

Trains/compares extra models on the frozen `c3_phase3_v5_repaired_v2` data. **Runtime > Change runtime type > T4 GPU** first.

Protocol: selection uses VALIDATION only; the held-out split is scored once, later, by `final_test_extra` after you approve the candidate list. Labels are tier-derived (not human-marked).

## 0. On your PC: zip the C3 folder (without the virtual environments)
PowerShell, from `ADAPTIVE_LEARN_SE/ai-services`:
```
tar -a -c -f c3.zip --exclude=.venv --exclude=.venv_c3 --exclude=__pycache__ --exclude=.env c3
```
Never upload `.env` (it holds API keys). Then upload `c3.zip` in the next cell.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable a GPU runtime')
from google.colab import files
up = files.upload()  # choose c3.zip
!unzip -q -o c3.zip -d /content
%cd /content/c3

In [ ]:
!pip -q install transformers sentence-transformers scikit-learn pandas numpy scipy sentencepiece tiktoken protobuf

## 1. Smoke test (1 minute, tiny model; numbers are NOT evidence)

In [ ]:
!python -m experiments.model_comparison_extra --smoke --out /content/out_smoke

## 2. Phase 1 - classical heads + 5 fine-tuned transformers x 3 seeds (validation only)
Roughly 5-15 minutes per transformer family on a T4 (estimate; DeBERTa is the slowest). Reduce `--seeds` or `--models` if the session is short.

In [ ]:
!python -m experiments.model_comparison_extra --out /content/out --save-dir /content/ckpt

## 3. Download results and checkpoints
Send `results_extra_phase1.json` back; keep `ckpt.zip` for phase 2 and for wiring the chosen model into the service.

In [ ]:
!cd /content && zip -q -r ckpt.zip ckpt && zip -q -r out.zip out
from google.colab import files
files.download('/content/out.zip'); files.download('/content/ckpt.zip')

## 4. Phase 2 - score the held-out split ONCE (only after you approve the candidate list)
Uncomment and run once. It refuses to run twice into the same folder.

In [ ]:
# !python -m experiments.final_test_extra --phase1 /content/out/results_extra_phase1.json --ckpt /content/ckpt --out /content/final